# 10_Corrective Actions & Decision Evaluation_v1

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# 10.01 — LOAD INPUTS
# ============================================================

PROJECT_DIR = Path(r"C:\ds4b\C2011-13 Wind Farm_v4")
OUTPUT_DIR = PROJECT_DIR / "07_Outputs"

# ------------------------------------------------------------
# 1. Verificar archivos de entrada
# ------------------------------------------------------------

required_files = [
    "cpm_activity_schedule.csv",
    "cpm_dependencies.csv",
    "cpm_project_summary.csv",
    "Scenario_Configuration.xlsx",
    "Scenario_Simulation_Results.xlsx"
]

missing_files = [
    f for f in required_files
    if not (OUTPUT_DIR / f).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Faltan archivos de entrada: {missing_files}"
    )

print("Archivos de entrada verificados.")

# ------------------------------------------------------------
# 2. Cargar datos congelados de NB_07
# ------------------------------------------------------------

df_baseline = pd.read_csv(
    OUTPUT_DIR / "cpm_activity_schedule.csv"
)

df_dependencies = pd.read_csv(
    OUTPUT_DIR / "cpm_dependencies.csv"
)

df_project_summary = pd.read_csv(
    OUTPUT_DIR / "cpm_project_summary.csv"
)

BASELINE_DURATION = float(
    df_project_summary.loc[0, "Project_Duration_CPM_Days"]
)

# ------------------------------------------------------------
# 3. Cargar configuración de NB_08
# ------------------------------------------------------------

df_activity_catalog = pd.read_excel(
    OUTPUT_DIR / "Scenario_Configuration.xlsx",
    sheet_name="Activity Catalog"
)

df_scenarios = pd.read_excel(
    OUTPUT_DIR / "Scenario_Configuration.xlsx",
    sheet_name="Scenarios"
)

# ------------------------------------------------------------
# 4. Cargar resultados de NB_09
# ------------------------------------------------------------

scenario_file = OUTPUT_DIR / "Scenario_Simulation_Results.xlsx"

df_scenario_summary = pd.read_excel(
    scenario_file,
    sheet_name="Scenario Summary"
)

df_activity_comparison = pd.read_excel(
    scenario_file,
    sheet_name="Activity Comparison"
)

df_scenario_cpm = pd.read_excel(
    scenario_file,
    sheet_name="Scenario CPM"
)

df_supply_impacts = pd.read_excel(
    scenario_file,
    sheet_name="Supply Impacts"
)

# ------------------------------------------------------------
# 5. Validaciones básicas
# ------------------------------------------------------------

assert len(df_baseline) == 135
assert len(df_dependencies) == 150
assert len(df_activity_catalog) == 135
assert BASELINE_DURATION == 299

assert df_scenario_summary["Scenario_ID"].nunique() == 3

print("\n========== INPUT VALIDATION ==========")
print(f"Baseline activities:       {len(df_baseline)}")
print(f"Dependencies:               {len(df_dependencies)}")
print(f"Activity catalog:           {len(df_activity_catalog)}")
print(f"Configured scenarios:       {len(df_scenarios)}")
print(f"Simulated scenarios:        {df_scenario_summary['Scenario_ID'].nunique()}")
print(f"Baseline CPM duration:      {BASELINE_DURATION:.0f} days")

# ------------------------------------------------------------
# 6. Inspeccionar estructura y actividades relevantes
# ------------------------------------------------------------

print("\n========== ACTIVITY CATALOG COLUMNS ==========")
print(df_activity_catalog.columns.tolist())

print("\n========== SCENARIO SUMMARY ==========")
display(df_scenario_summary)

print("\n========== SCENARIO CONFIGURATION ==========")
display(df_scenarios)

id_col = (
    "ID" if "ID" in df_activity_catalog.columns
    else "Activity_ID"
)

print("\n========== ACTIVITIES 79 AND 20 ==========")
display(
    df_activity_catalog[
        df_activity_catalog[id_col].isin([79, 20])
    ].T
)

print("\nNB_10 — Inputs loaded successfully.")

enemos los datos necesarios y ya aparecen dos situaciones muy distintas:

| Actividad | Situación | Implicación |
| :--- | :--- | :--- |
| **ID 79 — injection of tower** | Retraso de suministro existente de 35 días, buffer de 21 y alarma crítica. | Riesgo de suministro, pero 179 días de holgura CPM. |
| **ID 20 — Roads and hardstands** | Actividad crítica, duración de 14 días y holgura cero. | Cualquier incremento de duración afecta directamente al plazo del proyecto. |

Esto nos permite plantear dos tipos de intervención diferentes:
- ID 79: acelerar o recuperar suministro, reduciendo la exposición al riesgo.
- ID 20: recuperar parte del incremento hipotético de duración mediante una acción correctiva de ejecución.
  
Una precaución importante: los 179 días de holgura de la actividad 79 no eliminan su alarma de suministro. 

Tampoco vamos a traducir automáticamente días de retraso del material en retrasos del proyecto.

In [4]:
# ============================================================
# 10.02 — SUPPLY IMPACT REVIEW
# ============================================================

print("========== SUPPLY IMPACTS FROM NB_09 ==========")
display(df_supply_impacts)

print("\n========== ACTIVITY 79 — BASELINE SUPPLY DATA ==========")

display(
    df_activity_catalog.loc[
        df_activity_catalog["ID"] == 79,
        [
            "ID",
            "Name",
            "Material_Status",
            "Supply_Delay_Days",
            "Buffer_Days",
            "Material_Alarm"
        ]
    ]
)

print("\n========== ACTIVITY 20 — BASELINE SCHEDULE DATA ==========")

display(
    df_baseline.loc[
        df_baseline["ID"] == 20,
        [
            "ID",
            "Name",
            "Duration_Plan_Days",
            "Total_Float_Days",
            "Is_Critical",
            "CPM_Start_Date",
            "CPM_Finish_Date"
        ]
    ]
)

Hay una distinción importante en la actividad 79:
- Los 35 días son el retraso de suministro que ya figura en el catálogo.
- Los 15 días de SC01/SC03 son el impacto adicional introducido por el escenario.
- El buffer disponible es de 21 días. Por tanto, el retraso de suministro de partida ya supera ese buffer en 14 días.
No debemos presentar una reducción de los 15 días hipotéticos como si solucionase todo el problema del proveedor.

Para la actividad 20, en cambio, los 7 días adicionales sí afectan directamente al plazo: es una actividad crítica, sin holgura, y forma parte de la cadena crítica 12 → 18 → 20.

Propongo dos acciones, con eficacias explícitamente hipotéticas:

| Acción | Intervención simulada | Recuperación supuesta |
| :--- | :--- | :--- |
| **CA01 — Supplier Expediting** | Reducir el impacto adicional de suministro de 15 a 5 días. | 10 días de exposición |
| **CA02 — Execution Recovery** | Reducir el incremento de duración de ID20 de 7 a 2 días. | 5 días de ejecución |

La primera reduce riesgo de suministro, pero no altera automáticamente el CPM. 

La segunda sí modifica la duración de la actividad y, por tanto, el plazo del proyecto.

In [5]:
# ============================================================
# 10.02 — CORRECTIVE ACTION PLAN
# ============================================================

corrective_actions = [
    {
        "Action_ID": "CA01",
        "Action_Name": "Supplier Expediting",
        "Target_Activity_ID": 79,
        "Target_Scenario": "SC01; SC03",
        "Action_Type": "SUPPLY_MITIGATION",
        "Scenario_Impact_Before_Days": 15,
        "Scenario_Impact_After_Days": 5,
        "Recovered_Days": 10,
        "Baseline_Supply_Delay_Days": 35,
        "Buffer_Days": 21,
        "Residual_Total_Supply_Delay_Days": 40,
        "Residual_Buffer_Excess_Days": 19,
        "CPM_Impact": "None assumed",
        "Assumption": (
            "Hypothetical expediting intervention reduces "
            "the additional scenario supply delay by 10 days. "
            "Baseline supply delay remains unchanged."
        )
    },
    {
        "Action_ID": "CA02",
        "Action_Name": "Execution Recovery",
        "Target_Activity_ID": 20,
        "Target_Scenario": "SC02; SC03",
        "Action_Type": "ACTIVITY_RECOVERY",
        "Scenario_Impact_Before_Days": 7,
        "Scenario_Impact_After_Days": 2,
        "Recovered_Days": 5,
        "Baseline_Supply_Delay_Days": np.nan,
        "Buffer_Days": np.nan,
        "Residual_Total_Supply_Delay_Days": np.nan,
        "Residual_Buffer_Excess_Days": np.nan,
        "CPM_Impact": "Recalculate CPM",
        "Assumption": (
            "Hypothetical execution intervention reduces "
            "the additional activity duration by 5 days. "
            "The original planned duration remains unchanged."
        )
    }
]

df_corrective_actions = pd.DataFrame(corrective_actions)

print("========== CORRECTIVE ACTION PLAN ==========")
display(df_corrective_actions)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert df_corrective_actions["Action_ID"].is_unique

assert (
    df_corrective_actions.loc[
        df_corrective_actions["Action_ID"] == "CA01",
        "Recovered_Days"
    ].iloc[0] == 10
)

assert (
    df_corrective_actions.loc[
        df_corrective_actions["Action_ID"] == "CA02",
        "Recovered_Days"
    ].iloc[0] == 5
)

assert (
    df_corrective_actions.loc[
        df_corrective_actions["Action_ID"] == "CA01",
        "Residual_Buffer_Excess_Days"
    ].iloc[0] == 19
)

print("\nCorrective action assumptions validated.")

Vamos a simular tres situaciones:

1. SC01 + CA01: mitigación del retraso de suministro.

2. SC02 + CA02: recuperación de cinco días en la actividad crítica.

3. SC03 + CA01 + CA02: aplicación conjunta de ambas acciones.

In [7]:
def calculate_scenario_cpm(durations, execution_order):

    durations = {
        activity_id: float(duration)
        for activity_id, duration in durations.items()
    }

    es = {}
    ef = {}

    # --------------------------------------------------------
    # FORWARD PASS
    # --------------------------------------------------------
    for activity_id in execution_order:

        duration = durations[activity_id]
        constraints = []

        for edge in incoming[activity_id]:

            predecessor = edge["Predecessor"]
            relation = edge["Relation"]
            lag = edge["Lag"]

            if relation == "FS":
                candidate = ef[predecessor] + lag
            elif relation == "SS":
                candidate = es[predecessor] + lag
            elif relation == "FF":
                candidate = ef[predecessor] + lag - duration
            elif relation == "SF":
                candidate = es[predecessor] + lag - duration
            else:
                raise ValueError(f"Unknown relation: {relation}")

            constraints.append(candidate)

        es[activity_id] = max([0.0] + constraints)
        ef[activity_id] = es[activity_id] + duration

    project_duration = max(ef.values())

    # --------------------------------------------------------
    # BACKWARD PASS — CORRECTED
    # --------------------------------------------------------
    ls = {}
    lf = {}

    for activity_id in reversed(execution_order):

        duration = durations[activity_id]

        # Global project-finish constraint for every activity
        constraints = [project_duration - duration]

        for edge in outgoing[activity_id]:

            successor = edge["Successor"]
            relation = edge["Relation"]
            lag = edge["Lag"]

            if relation == "FS":
                candidate = ls[successor] - lag - duration
            elif relation == "SS":
                candidate = ls[successor] - lag
            elif relation == "FF":
                candidate = lf[successor] - lag - duration
            elif relation == "SF":
                candidate = lf[successor] - lag
            else:
                raise ValueError(f"Unknown relation: {relation}")

            constraints.append(candidate)

        ls[activity_id] = min(constraints)
        lf[activity_id] = ls[activity_id] + duration

    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------
    results = pd.DataFrame({
        "ID": list(durations.keys()),
        "Duration_Simulated_Days": list(durations.values())
    })

    results["ES"] = results["ID"].map(es)
    results["EF"] = results["ID"].map(ef)
    results["LS"] = results["ID"].map(ls)
    results["LF"] = results["ID"].map(lf)

    results["Total_Float"] = results["LS"] - results["ES"]

    results["Is_Critical"] = (
        results["Total_Float"].abs() <= 1e-6
    )

    return results, project_duration

In [8]:
from graphlib import TopologicalSorter

# ------------------------------------------------------------
# 1. Construir relaciones de entrada y salida
# ------------------------------------------------------------

incoming = {int(i): [] for i in df_baseline["ID"]}
outgoing = {int(i): [] for i in df_baseline["ID"]}

for row in df_dependencies.itertuples(index=False):

    predecessor = int(row.Predecessor_ID)
    successor = int(row.Successor_ID)

    edge = {
        "Predecessor": predecessor,
        "Successor": successor,
        "Relation": row.Relation_Type,
        "Lag": float(row.Lag)
    }

    incoming[successor].append(edge)
    outgoing[predecessor].append(edge)

# ------------------------------------------------------------
# 2. Orden topológico
# ------------------------------------------------------------

predecessors = {
    int(activity_id): {
        edge["Predecessor"]
        for edge in incoming[int(activity_id)]
    }
    for activity_id in df_baseline["ID"]
}

execution_order = list(
    TopologicalSorter(predecessors).static_order()
)

assert len(execution_order) == len(df_baseline)

# ------------------------------------------------------------
# 3. Duraciones baseline
# ------------------------------------------------------------

baseline_durations = (
    df_baseline
    .set_index("ID")["Duration_Plan_Days"]
    .astype(float)
    .to_dict()
)

# ------------------------------------------------------------
# 4. Ejecutar CPM
# ------------------------------------------------------------

df_cpm_validation, calculated_duration = calculate_scenario_cpm(
    baseline_durations,
    execution_order
)

# ------------------------------------------------------------
# 5. Comparar con NB_07
# ------------------------------------------------------------

df_validation = df_cpm_validation.merge(
    df_baseline[["ID", "ES_Days", "EF_Days",
                 "LS_Days", "LF_Days",
                 "Total_Float_Days"]],
    on="ID",
    how="left"
)

checks = {
    "ES": (df_validation["ES"] - df_validation["ES_Days"]).abs().max(),
    "EF": (df_validation["EF"] - df_validation["EF_Days"]).abs().max(),
    "LS": (df_validation["LS"] - df_validation["LS_Days"]).abs().max(),
    "LF": (df_validation["LF"] - df_validation["LF_Days"]).abs().max(),
    "Float": (
        df_validation["Total_Float"]
        - df_validation["Total_Float_Days"]
    ).abs().max()
}

print("========== CPM BASELINE VALIDATION ==========")
print(f"Reference duration: {BASELINE_DURATION:.2f}")
print(f"Calculated duration: {calculated_duration:.2f}")

for metric, error in checks.items():
    print(f"Maximum {metric} difference: {error:.6f}")

print(
    "\nCritical activities:",
    sorted(
        df_cpm_validation.loc[
            df_cpm_validation["Is_Critical"], "ID"
        ].tolist()
    )
)

assert abs(calculated_duration - BASELINE_DURATION) < 1e-6
assert all(error < 1e-6 for error in checks.values())

print("\nSUCCESS: CPM engine reproduces NB_07 baseline exactly.")

In [9]:
# ============================================================
# 10.03.03 — SIMULATE CORRECTIVE ACTIONS
# ============================================================

action_cpm_results = []
action_summary = []
action_supply_impacts = []

# Escenarios que reciben cada acción
scenarios_ca01 = ["SC01", "SC03"]
scenarios_ca02 = ["SC02", "SC03"]

# Impactos residuales definidos en el plan correctivo
residual_supply_delay = 5
residual_activity_delay = 2

# ------------------------------------------------------------
# 1. Simular cada escenario
# ------------------------------------------------------------

for scenario_id in sorted(df_scenarios["Scenario_ID"].unique()):

    scenario_rows = df_scenarios[
        df_scenarios["Scenario_ID"] == scenario_id
    ]

    durations = baseline_durations.copy()

    activity_delay_total = 0
    supply_delay_total = 0

    # Aplicar impactos y acciones correctivas
    for row in scenario_rows.itertuples(index=False):

        activity_id = int(row.Activity_ID)
        impact_type = row.Impact_Type
        original_impact = float(row.Delay_Days)

        if impact_type == "ACTIVITY_DELAY":

            remaining_impact = (
                residual_activity_delay
                if scenario_id in scenarios_ca02
                and activity_id == 20
                else original_impact
            )

            durations[activity_id] += remaining_impact
            activity_delay_total += remaining_impact

        elif impact_type == "SUPPLY_DELAY":

            remaining_impact = (
                residual_supply_delay
                if scenario_id in scenarios_ca01
                and activity_id == 79
                else original_impact
            )

            supply_delay_total += remaining_impact

        else:
            raise ValueError(
                f"Unknown impact type: {impact_type}"
            )

    # --------------------------------------------------------
    # 2. Ejecutar CPM
    # --------------------------------------------------------

    df_result, project_duration = calculate_scenario_cpm(
        durations,
        execution_order
    )

    df_result.insert(0, "Scenario_ID", scenario_id)
    action_cpm_results.append(df_result)

    critical_ids = sorted(
        df_result.loc[df_result["Is_Critical"], "ID"].tolist()
    )

    # --------------------------------------------------------
    # 3. Resumen de resultados
    # --------------------------------------------------------

    action_summary.append({
        "Scenario_ID": scenario_id,
        "Project_Duration_Days": project_duration,
        "Project_Delay_Days": project_duration - BASELINE_DURATION,
        "Activity_Delay_Days": activity_delay_total,
        "Supply_Delay_Days": supply_delay_total,
        "Critical_Activities": len(critical_ids),
        "Critical_Activity_IDs": critical_ids
    })

    # --------------------------------------------------------
    # 4. Impacto residual de suministro
    # --------------------------------------------------------

    for row in scenario_rows.itertuples(index=False):

        if row.Impact_Type == "SUPPLY_DELAY":

            baseline_supply = float(
                df_activity_catalog.loc[
                    df_activity_catalog["ID"] == int(row.Activity_ID),
                    "Supply_Delay_Days"
                ].iloc[0]
            )

            buffer_days = float(
                df_activity_catalog.loc[
                    df_activity_catalog["ID"] == int(row.Activity_ID),
                    "Buffer_Days"
                ].iloc[0]
            )

            action_supply_impacts.append({
                "Scenario_ID": scenario_id,
                "Activity_ID": int(row.Activity_ID),
                "Baseline_Supply_Delay_Days": baseline_supply,
                "Scenario_Supply_Impact_Days": float(row.Delay_Days),
                "Residual_Scenario_Impact_Days": (
                    residual_supply_delay
                    if scenario_id in scenarios_ca01
                    else float(row.Delay_Days)
                ),
                "Residual_Total_Supply_Delay_Days": (
                    baseline_supply
                    + (
                        residual_supply_delay
                        if scenario_id in scenarios_ca01
                        else float(row.Delay_Days)
                    )
                ),
                "Buffer_Days": buffer_days,
                "Residual_Buffer_Excess_Days": (
                    baseline_supply
                    + (
                        residual_supply_delay
                        if scenario_id in scenarios_ca01
                        else float(row.Delay_Days)
                    )
                    - buffer_days
                )
            })

# ------------------------------------------------------------
# 5. Consolidar resultados
# ------------------------------------------------------------

df_action_cpm = pd.concat(
    action_cpm_results,
    ignore_index=True
)

df_action_summary = pd.DataFrame(action_summary)

df_action_supply = pd.DataFrame(action_supply_impacts)

# ------------------------------------------------------------
# 6. Mostrar resultados
# ------------------------------------------------------------

print("========== CORRECTED SCENARIO SUMMARY ==========")
display(df_action_summary)

print("\n========== RESIDUAL SUPPLY IMPACTS ==========")
display(df_action_supply)

print("\n========== CRITICAL ACTIVITIES ==========")

for scenario_id in sorted(df_action_cpm["Scenario_ID"].unique()):

    critical_ids = sorted(
        df_action_cpm.loc[
            (df_action_cpm["Scenario_ID"] == scenario_id)
            & df_action_cpm["Is_Critical"],
            "ID"
        ].tolist()
    )

    print(f"{scenario_id}: {critical_ids}")

# ------------------------------------------------------------
# 7. Validaciones
# ------------------------------------------------------------

expected_durations = {
    "SC01": 299,
    "SC02": 301,
    "SC03": 301
}

for scenario_id, expected in expected_durations.items():

    actual = float(
        df_action_summary.loc[
            df_action_summary["Scenario_ID"] == scenario_id,
            "Project_Duration_Days"
        ].iloc[0]
    )

    assert abs(actual - expected) < 1e-6, (
        f"{scenario_id}: expected {expected}, got {actual}"
    )

assert df_action_summary["Scenario_ID"].nunique() == 3

print("\nSUCCESS: Corrective action simulations validated.")

| Indicador | Sin acciones (NB_09) | Con acciones (NB_10) |
| :--- | :--- | :--- |
| **SC01 — Duración proyecto** | 299 días | 299 días |
| **SC02 — Duración proyecto** | 306 días | 301 días |
| **SC03 — Duración proyecto** | 306 días | 301 días |
| **SC02/SC03 — Retraso del proyecto** | 7 días | 2 días |
| **SC01/SC03 — Impacto adicional de suministro** | 15 días | 5 días |

Dos conclusiones:
- CA02 recupera 5 de los 7 días de retraso de ejecución: una recuperación del 71,4 %.
- CA01 reduce en 10 días el impacto adicional de suministro, pero la exposición total queda en 40 días frente a un buffer de 21. Persisten 19 días de exceso.
La cadena crítica continúa siendo 12 → 18 → 20. No hemos cambiado artificialmente la ruta crítica.

Pasamos al último bloque: evaluar, comparar y exportar.

**Decision Evaluation & Export**

Esta última celda construye cuatro salidas:
    
1. Comparativa de escenarios antes y después de las acciones.

2. Evaluación de las acciones correctivas.

3. Riesgo residual de suministro.

4. Comparativa CPM por actividad.

Además, exporta un Excel consolidado para utilizarlo posteriormente en el informe ejecutivo.

In [11]:
# ============================================================
# 10.04 — DECISION EVALUATION & EXPORT
# ============================================================

# ------------------------------------------------------------
# 1. Comparación de escenarios: antes y después
# ------------------------------------------------------------

df_decision_summary = (
    df_scenario_summary[
        [
            "Scenario_ID",
            "Project_Duration_Days",
            "Project_Delay_Days",
            "Activity_Delay_Days",
            "Supply_Delay_Days"
        ]
    ]
    .rename(columns={
        "Project_Duration_Days": "Duration_Before_Actions",
        "Project_Delay_Days": "Delay_Before_Actions",
        "Activity_Delay_Days": "Activity_Impact_Before",
        "Supply_Delay_Days": "Supply_Impact_Before"
    })
    .merge(
        df_action_summary[
            [
                "Scenario_ID",
                "Project_Duration_Days",
                "Project_Delay_Days",
                "Activity_Delay_Days",
                "Supply_Delay_Days",
                "Critical_Activity_IDs"
            ]
        ].rename(columns={
            "Project_Duration_Days": "Duration_After_Actions",
            "Project_Delay_Days": "Delay_After_Actions",
            "Activity_Delay_Days": "Activity_Impact_After",
            "Supply_Delay_Days": "Supply_Impact_After"
        }),
        on="Scenario_ID",
        how="inner"
    )
)

df_decision_summary["Project_Days_Recovered"] = (
    df_decision_summary["Duration_Before_Actions"]
    - df_decision_summary["Duration_After_Actions"]
)

df_decision_summary["Delay_Reduction_Pct"] = np.where(
    df_decision_summary["Delay_Before_Actions"] > 0,
    100 * (
        df_decision_summary["Project_Days_Recovered"]
        / df_decision_summary["Delay_Before_Actions"]
    ),
    0
)

# ------------------------------------------------------------
# 2. Evaluación de las acciones
# ------------------------------------------------------------

df_action_evaluation = df_corrective_actions[
    [
        "Action_ID",
        "Action_Name",
        "Target_Activity_ID",
        "Action_Type",
        "Scenario_Impact_Before_Days",
        "Scenario_Impact_After_Days",
        "Recovered_Days",
        "CPM_Impact",
        "Assumption"
    ]
].copy()

df_action_evaluation["Recovery_Rate_Pct"] = (
    100
    * df_action_evaluation["Recovered_Days"]
    / df_action_evaluation["Scenario_Impact_Before_Days"]
)

# ------------------------------------------------------------
# 3. Evaluación del riesgo residual de suministro
# ------------------------------------------------------------

df_supply_evaluation = df_action_supply.copy()

df_supply_evaluation["Original_Total_Supply_Delay_Days"] = (
    df_supply_evaluation["Baseline_Supply_Delay_Days"]
    + df_supply_evaluation["Scenario_Supply_Impact_Days"]
)

df_supply_evaluation["Original_Buffer_Excess_Days"] = (
    df_supply_evaluation["Original_Total_Supply_Delay_Days"]
    - df_supply_evaluation["Buffer_Days"]
)

df_supply_evaluation["Supply_Risk_Reduction_Days"] = (
    df_supply_evaluation["Original_Total_Supply_Delay_Days"]
    - df_supply_evaluation["Residual_Total_Supply_Delay_Days"]
)

df_supply_evaluation["Residual_Risk_Status"] = np.where(
    df_supply_evaluation["Residual_Buffer_Excess_Days"] > 0,
    "BUFFER EXCEEDED",
    "WITHIN BUFFER"
)

# ------------------------------------------------------------
# 4. Comparación CPM por actividad
# ------------------------------------------------------------

cpm_columns = [
    "Scenario_ID",
    "ID",
    "Duration_Simulated_Days",
    "ES",
    "EF",
    "LS",
    "LF",
    "Total_Float",
    "Is_Critical"
]

assert all(
    col in df_scenario_cpm.columns
    for col in cpm_columns
), "Unexpected NB_09 Scenario CPM structure."

assert all(
    col in df_action_cpm.columns
    for col in cpm_columns
), "Unexpected NB_10 Scenario CPM structure."

df_cpm_before = df_scenario_cpm[cpm_columns].copy()

df_cpm_after = df_action_cpm[cpm_columns].copy()

df_cpm_comparison = df_cpm_before.merge(
    df_cpm_after,
    on=["Scenario_ID", "ID"],
    suffixes=("_Before", "_After"),
    how="inner"
)

for metric in [
    "Duration_Simulated_Days",
    "ES",
    "EF",
    "LS",
    "LF",
    "Total_Float"
]:
    df_cpm_comparison[f"{metric}_Change"] = (
        df_cpm_comparison[f"{metric}_After"]
        - df_cpm_comparison[f"{metric}_Before"]
    )

df_cpm_comparison["Critical_Status_Changed"] = (
    df_cpm_comparison["Is_Critical_Before"]
    != df_cpm_comparison["Is_Critical_After"]
)

# ------------------------------------------------------------
# 5. Mostrar evaluación
# ------------------------------------------------------------

print("========== DECISION EVALUATION ==========")
display(df_decision_summary)

print("\n========== CORRECTIVE ACTION EVALUATION ==========")
display(df_action_evaluation)

print("\n========== RESIDUAL SUPPLY RISK ==========")
display(df_supply_evaluation)

print("\n========== CPM IMPACT SUMMARY ==========")

display(
    df_cpm_comparison.groupby("Scenario_ID").agg(
        Activities_Compared=("ID", "count"),
        Activities_With_Date_Changes=(
            "ES_Change",
            lambda x: int(
                (
                    (x.abs() > 1e-6)
                    | (
                        df_cpm_comparison.loc[
                            x.index, "EF_Change"
                        ].abs() > 1e-6
                    )
                ).sum()
            )
        ),
        Critical_Status_Changes=(
            "Critical_Status_Changed", "sum"
        )
    ).reset_index()
)

# ------------------------------------------------------------
# 6. Validaciones finales
# ------------------------------------------------------------

assert len(df_decision_summary) == 3
assert len(df_action_evaluation) == 2
assert len(df_supply_evaluation) == 2
assert len(df_cpm_comparison) == 405

assert (
    df_decision_summary["Project_Days_Recovered"].tolist()
    == [0.0, 5.0, 5.0]
)

assert (
    df_supply_evaluation["Residual_Buffer_Excess_Days"] == 19
).all()

assert not df_cpm_comparison["Critical_Status_Changed"].any()

# ------------------------------------------------------------
# 7. Exportación
# ------------------------------------------------------------

output_excel = OUTPUT_DIR / "Corrective_Action_Evaluation.xlsx"

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:

    df_decision_summary.to_excel(
        writer, sheet_name="Decision Summary", index=False
    )

    df_action_evaluation.to_excel(
        writer, sheet_name="Corrective Actions", index=False
    )

    df_supply_evaluation.to_excel(
        writer, sheet_name="Supply Risk", index=False
    )

    df_cpm_comparison.to_excel(
        writer, sheet_name="Activity CPM Comparison", index=False
    )

df_decision_summary.to_csv(
    OUTPUT_DIR / "Corrective_Action_Summary.csv",
    index=False
)

# ------------------------------------------------------------
# 8. Verificación de archivos
# ------------------------------------------------------------

assert output_excel.exists()

check_sheets = pd.ExcelFile(output_excel).sheet_names

assert len(check_sheets) == 4

print("\n========== EXPORT VALIDATION ==========")
print(f"Excel: {output_excel}")
print(f"Sheets: {check_sheets}")
print("CSV: Corrective_Action_Summary.csv")

print("\nSUCCESS: NB_10 completed successfully.")

| Indicador | Resultado |
| :--- | :--- |
| **Escenarios evaluados** | 3 |
| **Acciones correctivas simuladas** | 2 |
| **Recuperación de plazo en SC02** | 5 días |
| **Recuperación de plazo en SC03** | 5 días |
| **Reducción del impacto adicional de suministro** | 10 días |
| **Cambios de ruta crítica** | 0 |
| **Validaciones CPM** | Correctas |

In [13]:
# ============================================================
# NB_10 — FINAL EXPORT & VALIDATION
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. OUTPUT PATHS
# ------------------------------------------------------------

OUTPUT_DIR = Path(r"C:\ds4b\C2011-13 Wind Farm_v4\07_Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCEL_PATH = OUTPUT_DIR / "Corrective_Action_Evaluation.xlsx"
CSV_PATH = OUTPUT_DIR / "Corrective_Action_Summary.csv"

# ------------------------------------------------------------
# 2. EXPORT EXCEL
# ------------------------------------------------------------

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:

    df_decision_summary.to_excel(
        writer, sheet_name="Decision Summary", index=False
    )

    df_action_evaluation.to_excel(
        writer, sheet_name="Corrective Actions", index=False
    )

    df_supply_evaluation.to_excel(
        writer, sheet_name="Supply Risk", index=False
    )

    df_cpm_comparison.to_excel(
        writer, sheet_name="Activity CPM Comparison", index=False
    )

# ------------------------------------------------------------
# 3. EXPORT CSV
# ------------------------------------------------------------

df_decision_summary.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 4. VALIDATION
# ------------------------------------------------------------

print("=" * 65)
print("NB_10 — FINAL EXPORT VALIDATION")
print("=" * 65)

expected_files = [EXCEL_PATH, CSV_PATH]

for file in expected_files:
    assert file.exists(), f"Missing file: {file}"
    assert file.stat().st_size > 0, f"Empty file: {file}"
    print(f"OK | {file.name} | {file.stat().st_size:,} bytes")

# Verify Excel structure
xls = pd.ExcelFile(EXCEL_PATH)

expected_sheets = [
    "Decision Summary",
    "Corrective Actions",
    "Supply Risk",
    "Activity CPM Comparison"
]

assert xls.sheet_names == expected_sheets

print("\nExcel sheets:")
for sheet in xls.sheet_names:
    df_check = pd.read_excel(EXCEL_PATH, sheet_name=sheet)
    print(f"  {sheet}: {len(df_check):,} rows")

# Verify CSV
df_csv_check = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

assert len(df_csv_check) == len(df_decision_summary)

print(f"\nCSV rows: {len(df_csv_check):,}")

print("\n" + "=" * 65)
print("NB_10 COMPLETED — OUTPUTS VERIFIED")
print("=" * 65)